# Test — LeakyBucketLocal (Australia / camelsaus_102101A)

Quick end-to-end test of `LeakyBucketLocal` using the ERA5 forcing for the
Australian catchment **camelsaus_102101A**.  
Observed discharge from the Caravan dataset is loaded for visual comparison.

Steps:
1. Load settings & forcing
2. Run the model for a range of `leakiness` values
3. Plot modelled vs observed discharge

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from rich import print

# Make sure the scripts folder is importable both on HPC and locally
try:
    from scripts.leakybucket_model import LeakyBucketLocal
except ImportError:
    project_root = Path().resolve().parent
    sys.path.insert(0, str(project_root))
    from scripts.leakybucket_model import LeakyBucketLocal

import ewatercycle
import ewatercycle.forcing

## 1. Settings & forcing

In [ ]:
settings_file = Path('../regions/australia/camelsaus_102101A/settings.json')
with open(settings_file) as f:
    settings = json.load(f)

print(f"Catchment : [cyan]{settings['caravan_id']}[/cyan]")
print(f"Calibration: {settings['calibration_start_date']} → {settings['calibration_end_date']}")
print(f"Validation : {settings['validation_start_date']} → {settings['validation_end_date']}")

In [ ]:
# ERA5 forcing — stored one level deeper than path_ERA5
era5_dir = Path(settings['path_ERA5']) / 'work' / 'diagnostic' / 'script'
era5_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(directory=era5_dir)
display(era5_forcing)

In [ ]:
# Quick look at the precipitation input
pr = xr.open_dataset(era5_forcing.directory / era5_forcing['pr'])['pr']
pr.plot(figsize=(12, 3), color='steelblue')
plt.title('ERA5 precipitation — camelsaus_102101A')
plt.ylabel('pr (kg m⁻² s⁻¹)')
plt.tight_layout()
plt.show()

In [ ]:
# Observed discharge from Caravan (for comparison)
caravan_forcing = ewatercycle.forcing.sources['CaravanForcing'].load(
    directory=settings['path_caravan']
)
obs_q = xr.open_dataset(
    caravan_forcing.directory / caravan_forcing['Q']
)['Q'].to_series()
obs_q.name = 'Observed (Caravan)'
print(f"Observed discharge: {len(obs_q)} days, "
      f"{obs_q.index[0].date()} → {obs_q.index[-1].date()}")

## 2. Run LeakyBucketLocal

Run for several `leakiness` values to explore the model's sensitivity.
The bucket equation is `Q = leakiness × S`, so a higher value drains faster.

In [ ]:
LEAKINESS_VALUES = [0.1, 0.3, 0.5, 1.0]  # d⁻¹
INITIAL_STORAGE  = 0.0                    # m

model_output = {}

for leakiness in LEAKINESS_VALUES:
    label = f'leakiness={leakiness}'
    print(f'Running [cyan]{label}[/cyan] ...')

    model = LeakyBucketLocal(forcing=era5_forcing)
    config_file, _ = model.setup(leakiness=leakiness, initial_storage=INITIAL_STORAGE)
    model.initialize(config_file)

    q_vals, timestamps = [], []
    while model.time < model.end_time:
        model.update()
        q_vals.append(model.get_value('discharge', dest=np.zeros(1))[0])
        timestamps.append(pd.Timestamp(model.time_as_datetime))

    model.finalize()

    model_output[label] = pd.Series(data=q_vals, index=timestamps, name=label)
    print(f'  → {len(q_vals)} timesteps, '
          f'mean Q = {np.mean(q_vals):.4f} m/d')

print('[bold green]All runs complete.[/bold green]')

## 3. Plot — modelled discharge vs observed

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# ── Top: all leakiness runs ───────────────────────────────────────────────────
colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(LEAKINESS_VALUES)))
for (label, series), color in zip(model_output.items(), colors):
    axes[0].plot(series.index, series.values, label=label, color=color,
                 linewidth=1.2, alpha=0.9)

axes[0].set_ylabel('Discharge (m d⁻¹)', fontsize=10)
axes[0].set_title('LeakyBucketLocal — ERA5 forcing (camelsaus_102101A)', fontsize=11)
axes[0].legend(fontsize=9, framealpha=0.8)
axes[0].grid(True, alpha=0.3)

# ── Bottom: best run vs observed ─────────────────────────────────────────────
# Pick the run with leakiness=0.5 as a reference
ref_label = 'leakiness=0.5'
ref_series = model_output[ref_label]

# Align observed to modelled time range
obs_aligned = obs_q.reindex(ref_series.index)

axes[1].plot(ref_series.index, ref_series.values, label=ref_label,
             color='steelblue', linewidth=1.2)
axes[1].plot(obs_aligned.index, obs_aligned.values, label='Observed (Caravan)',
             color='black', linewidth=0.8, alpha=0.7)
axes[1].set_ylabel('Discharge (m d⁻¹)', fontsize=10)
axes[1].set_xlabel('Date', fontsize=10)
axes[1].legend(fontsize=9, framealpha=0.8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Performance metrics (KGE / NSE)

Quick check against the calibration period used in `settings.json`.

In [ ]:
def kge(sim, obs):
    """Kling-Gupta Efficiency."""
    mask = ~(np.isnan(sim) | np.isnan(obs))
    s, o = sim[mask], obs[mask]
    r = np.corrcoef(s, o)[0, 1]
    alpha = s.std() / o.std()
    beta  = s.mean() / o.mean()
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def nse(sim, obs):
    """Nash-Sutcliffe Efficiency."""
    mask = ~(np.isnan(sim) | np.isnan(obs))
    s, o = sim[mask], obs[mask]
    return 1 - np.sum((s - o)**2) / np.sum((o - o.mean())**2)


cal_start = settings['calibration_start_date'][:10]
cal_end   = settings['calibration_end_date'][:10]

rows = []
for label, series in model_output.items():
    sim = series[cal_start:cal_end]
    obs = obs_q.reindex(sim.index)
    rows.append({
        'run':   label,
        'KGE':   kge(sim.values, obs.values),
        'NSE':   nse(sim.values, obs.values),
        'mean_sim (m/d)': sim.mean(),
        'mean_obs (m/d)': obs.mean(),
    })

metrics = pd.DataFrame(rows).set_index('run')
display(metrics.style.format('{:.4f}').highlight_max(
    subset=['KGE', 'NSE'], color='#c8f7c5'
).set_caption(f'Calibration period: {cal_start} → {cal_end}'))

## 5. Inspect model state & parameters

In [ ]:
# Re-run a single model just to inspect its parameter and state interface
model = LeakyBucketLocal(forcing=era5_forcing)
config_file, cfg_dir = model.setup(leakiness=0.5, initial_storage=0.01)
model.initialize(config_file)

print('[bold]Parameters:[/bold]')
for k, v in model.parameters:
    print(f'  {k} = {v}')

print('[bold]States (at t=0 after initialize):[/bold]')
for k, v in model.states:
    print(f'  {k} = {v}')

print(f'[bold]Time:[/bold] {model.time_as_isostr}  →  end: {model.end_time_as_isostr}')

# Take one step and check state again
model.update()
print('[bold]States (after first update):[/bold]')
for k, v in model.states:
    print(f'  {k} = {v}')

model.finalize()